# Notebook IV.3 — Primer armónico contra balance multiarmónico

Este notebook acompaña la **Ventana computacional IV.3** del Capítulo IV.

Su propósito es comparar aproximaciones con **uno, tres y cinco armónicos** para el oscilador de Duffing forzado.  
La idea central es mostrar que la no linealidad puede interpretarse como un mecanismo de **acoplamiento espectral**: cuando la respuesta deja de ser casi sinusoidal, el primer armónico conserva valor pedagógico, pero ya no basta para describir con fidelidad la órbita periódica.

## Ejecutar este notebook en Google Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricomelgozajjesus/monografia-armonica/blob/main/python-lab/notebooks/Notebook_IV_03_Multiarmonico.ipynb)

## 1. Modelo

Trabajaremos de nuevo con la ecuación de Duffing forzado:

$$
\ddot{x} + 2\zeta \omega_0 \dot{x} + \omega_0^2 x + \alpha x^3 = F\cos(\Omega t).
$$

A diferencia del notebook anterior, aquí usaremos una representación armónica truncada con varios armónicos impares:

$$
x(t) \approx \sum_{k\in\{1,3,5,\dots\}} \big(a_k\cos(k\Omega t)+b_k\sin(k\Omega t)\big).
$$

Después construiremos el residual en el dominio temporal, lo proyectaremos numéricamente sobre la base trigonométrica y resolveremos el sistema no lineal con Newton.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

## 2. Parámetros

Puedes comenzar con estos valores y luego modificarlos para explorar regímenes más o menos no lineales.

In [ ]:
zeta = 0.05
omega0 = 1.0
alpha = 1.0
F = 0.35
Omega = 1.0

harmonic_sets = {
    "1 armónico": [1],
    "3 armónicos": [1, 3, 5],
    "5 armónicos": [1, 3, 5, 7, 9],
}

## 3. Base armónica y reconstrucción temporal

Representaremos el vector de incógnitas como

$$
c = [a_{k_1}, b_{k_1}, a_{k_2}, b_{k_2}, \dots]^T.
$$

Para una malla temporal uniforme dentro de un periodo, construiremos la señal aproximada, su derivada y su segunda derivada.

In [ ]:
def build_time_grid(Omega, n_points=1024):
    T = 2*np.pi / Omega
    t = np.linspace(0.0, T, n_points, endpoint=False)
    return t, T

def unpack_coeffs(c, harmonics):
    coeffs = {}
    for i, k in enumerate(harmonics):
        coeffs[k] = (c[2*i], c[2*i + 1])
    return coeffs

def reconstruct_signal(c, harmonics, t, Omega):
    x = np.zeros_like(t, dtype=float)
    xd = np.zeros_like(t, dtype=float)
    xdd = np.zeros_like(t, dtype=float)
    coeffs = unpack_coeffs(c, harmonics)

    for k in harmonics:
        a, b = coeffs[k]
        theta = k * Omega * t
        cosk = np.cos(theta)
        sink = np.sin(theta)

        x += a*cosk + b*sink
        xd += -a*k*Omega*sink + b*k*Omega*cosk
        xdd += -(k*Omega)**2 * (a*cosk + b*sink)
    return x, xd, xdd

## 4. Residual proyectado

Definimos el residual temporal

$$
r(t)=\ddot{x}(t)+2\zeta\omega_0\dot{x}(t)+\omega_0^2x(t)+\alpha x^3(t)-F\cos(\Omega t).
$$

Luego lo proyectamos numéricamente sobre cada función base retenida:

$$
\langle r,\cos(k\Omega t)\rangle,\qquad \langle r,\sin(k\Omega t)\rangle.
$$

Así obtenemos un sistema algebraico no lineal para los coeficientes armónicos.

In [ ]:
def harmonic_residual(c, harmonics, zeta, omega0, alpha, F, Omega, n_points=2048):
    t, T = build_time_grid(Omega, n_points=n_points)
    x, xd, xdd = reconstruct_signal(c, harmonics, t, Omega)
    forcing = F * np.cos(Omega * t)

    r = xdd + 2*zeta*omega0*xd + omega0**2 * x + alpha * x**3 - forcing

    R = []
    for k in harmonics:
        ck = np.cos(k * Omega * t)
        sk = np.sin(k * Omega * t)

        # Proyección trapezoidal/rectangular sobre un periodo uniforme
        Rc = (2.0 / len(t)) * np.sum(r * ck)
        Rs = (2.0 / len(t)) * np.sum(r * sk)
        R.extend([Rc, Rs])

    return np.array(R, dtype=float)

def numerical_jacobian(fun, c, eps=1e-7):
    c = np.array(c, dtype=float)
    f0 = fun(c)
    J = np.zeros((len(f0), len(c)), dtype=float)
    for j in range(len(c)):
        cp = c.copy()
        cm = c.copy()
        cp[j] += eps
        cm[j] -= eps
        fp = fun(cp)
        fm = fun(cm)
        J[:, j] = (fp - fm) / (2*eps)
    return J

## 5. Newton–Raphson para el sistema multiarmónico

Usaremos un jacobiano numérico para mantener el notebook claro y flexible.  
Esto no es lo más eficiente, pero sí muy pedagógico.

In [ ]:
def newton_multi(fun, c0, tol=1e-10, max_iter=25):
    c = np.array(c0, dtype=float)
    history = []

    for k in range(max_iter):
        R = fun(c)
        nr = np.linalg.norm(R)
        history.append((k, c.copy(), nr))
        if nr < tol:
            return c, history, True
        J = numerical_jacobian(fun, c)
        delta = np.linalg.solve(J, -R)
        c = c + delta

    R = fun(c)
    history.append((max_iter, c.copy(), np.linalg.norm(R)))
    return c, history, False

## 6. Elegir semillas iniciales

Usaremos una semilla sencilla: solo una componente cosenoidal en el armónico fundamental.  
En algunos casos exigentes, conviene usar como semilla la solución del caso con menos armónicos.

In [ ]:
def initial_guess(harmonics, amplitude_guess=0.8):
    c0 = np.zeros(2 * len(harmonics), dtype=float)
    c0[0] = amplitude_guess   # a_1
    return c0

## 7. Resolver los casos con uno, tres y cinco armónicos

In [ ]:
solutions = {}
histories = {}

for label, harmonics in harmonic_sets.items():
    fun = lambda c, hs=harmonics: harmonic_residual(c, hs, zeta, omega0, alpha, F, Omega, n_points=2048)
    c0 = initial_guess(harmonics, amplitude_guess=0.7)
    sol, hist, converged = newton_multi(fun, c0, tol=1e-10, max_iter=20)
    solutions[label] = {
        "harmonics": harmonics,
        "coeffs": sol,
        "converged": converged,
        "residual_norm": hist[-1][2],
    }
    histories[label] = hist

for label, info in solutions.items():
    print(label)
    print("  armónicos:", info["harmonics"])
    print("  convergió:", info["converged"])
    print("  norma final del residual:", info["residual_norm"])
    print()

## 8. Comparar convergencia

In [ ]:
plt.figure(figsize=(8.5, 4.8))
for label, hist in histories.items():
    iters = [row[0] for row in hist]
    norms = [row[2] for row in hist]
    plt.semilogy(iters, norms, marker="o", label=label)

plt.xlabel("Iteración")
plt.ylabel("Norma del residual")
plt.title("Convergencia de Newton en distintas truncaciones armónicas")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Reconstrucción temporal de las soluciones

Ahora reconstruimos las señales aproximadas sobre varios periodos y las comparamos entre sí.

In [ ]:
t_long = np.linspace(0, 4*(2*np.pi/Omega), 3000)

plt.figure(figsize=(10, 4.8))
for label, info in solutions.items():
    x, xd, xdd = reconstruct_signal(info["coeffs"], info["harmonics"], t_long, Omega)
    plt.plot(t_long, x, label=label)

plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Comparación temporal: 1, 3 y 5 armónicos")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10. Espectros reconstruidos

Aunque cada truncación tiene un número distinto de armónicos retenidos, podemos representar sus amplitudes espectrales para comparar la energía distribuida entre frecuencias.

In [ ]:
def amplitude_spectrum_from_coeffs(c, harmonics):
    amps = {}
    coeffs = unpack_coeffs(c, harmonics)
    for k in harmonics:
        a, b = coeffs[k]
        amps[k] = np.sqrt(a*a + b*b)
    return amps

all_orders = [1, 3, 5, 7, 9]

plt.figure(figsize=(9, 4.8))
width = 0.22
xpos = np.arange(len(all_orders))

for j, (label, info) in enumerate(solutions.items()):
    amps = amplitude_spectrum_from_coeffs(info["coeffs"], info["harmonics"])
    y = [amps.get(k, 0.0) for k in all_orders]
    plt.bar(xpos + j*width - width, y, width=width, label=label)

plt.xticks(xpos, [str(k) for k in all_orders])
plt.xlabel("Orden armónico")
plt.ylabel("Amplitud")
plt.title("Contenido armónico reconstruido")
plt.grid(True, axis="y", alpha=0.3)
plt.legend()
plt.show()

## 11. Comparación con integración temporal directa

Para dar contexto físico, resolvemos el sistema original en el tiempo y extraemos su respuesta estacionaria.  
Luego la superponemos con las aproximaciones armónicas.

In [ ]:
def duffing_rhs(t, y, zeta, omega0, alpha, F, Omega):
    x, v = y
    dxdt = v
    dvdt = -2*zeta*omega0*v - omega0**2*x - alpha*x**3 + F*np.cos(Omega*t)
    return [dxdt, dvdt]

T = 2*np.pi / Omega
t_eval = np.linspace(0, 250*T, 250*400 + 1)

sol_ivp = solve_ivp(
    duffing_rhs,
    (0.0, 250*T),
    [0.0, 0.0],
    t_eval=t_eval,
    args=(zeta, omega0, alpha, F, Omega),
    rtol=1e-8,
    atol=1e-10,
)

t_full = sol_ivp.t
x_full = sol_ivp.y[0]

samples_keep = 40*400
t_ss = t_full[-samples_keep:]
x_ss = x_full[-samples_keep:]
t_ss_shift = t_ss - t_ss[0]

In [ ]:
plt.figure(figsize=(10, 4.8))
plt.plot(t_ss_shift, x_ss, linewidth=2.0, label="Integración temporal")

for label, info in solutions.items():
    xh, _, _ = reconstruct_signal(info["coeffs"], info["harmonics"], t_ss_shift, Omega)
    plt.plot(t_ss_shift, xh, "--", label=label)

plt.xlim(0, 8*T)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Comparación con la respuesta temporal estacionaria")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 12. Error RMS respecto de la simulación temporal

Medimos ahora, de forma sencilla, qué tan cerca queda cada truncación armónica de la respuesta temporal obtenida por integración directa.

In [ ]:
for label, info in solutions.items():
    xh, _, _ = reconstruct_signal(info["coeffs"], info["harmonics"], t_ss_shift, Omega)
    err = np.sqrt(np.mean((xh - x_ss)**2))
    print(f"{label:12s} -> error RMS = {err:.6e}")

## 13. Interpretación

Este notebook ilustra varias ideas importantes:

- el **primer armónico** puede capturar la estructura dominante de la respuesta;
- al aumentar la no linealidad o la cercanía a resonancia, aparecen armónicos superiores con amplitud apreciable;
- el balance **multiarmónico** permite absorber mejor esa redistribución espectral;
- la no linealidad puede leerse como un mecanismo de **acoplamiento entre frecuencias**.

Así, el paso de una truncación simple a una más rica no es una complicación gratuita, sino una respuesta natural a la geometría espectral de la dinámica.

## 14. Exploraciones sugeridas

1. Incrementa `F` y observa cuándo el caso de un solo armónico deja de ser razonable.
2. Cambia `alpha` y compara hardening y softening.
3. Aumenta el número de armónicos retenidos en `harmonic_sets`.
4. Usa como semilla de un caso rico la solución de uno más pobre.
5. Observa cómo cambia el error RMS al variar la truncación.

## 15. Hacia el siguiente notebook

El siguiente paso natural es estudiar el caso en que la **frecuencia también es desconocida** y debe resolverse junto con los coeficientes armónicos, introduciendo una **condición de fase** para eliminar la indeterminación por traslación temporal.